Basic Details Extractions

In [1]:
import time
import numpy as np
import pandas as pd
import os
import random
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

In [ ]:
chrome_options = Options()
chrome_options.add_argument("--disable-http2")
chrome_options.add_argument("--incognito")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_argument("--ignore-certificate-errors")
chrome_options.add_argument("--enable-features=NetworkServiceInProcess")
chrome_options.add_argument("--disable-features=NetworkService")
chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.63 Safari/537.36"
)

driver = webdriver.Chrome(options=chrome_options)
url = "https://www.99acres.com/"
driver.get(url)

# Explicit Wait

wait = WebDriverWait(driver, 5)


def wait_for_page_to_load(driver, wait_obj):
    
    """Wait until the page is fully loaded."""
    title = driver.title
    try:
        wait_obj.until(lambda d: d.execute_script("return document.readyState") == "complete")
    except Exception:
        print(f'The webpage "{title}" did not get fully loaded.\n')
    else:
        print(f'The webpage "{title}" did get fully loaded.\n')


def handle_popup():
    """Close known occasional popups if present."""
    
    try:
        popup_button = driver.find_element(By.XPATH, '//*[@id="app"]/div/div[3]/div/div[2]/button')
        popup_button.click()
        time.sleep(2)
    except NoSuchElementException:
        pass


wait_for_page_to_load(driver, wait)

# Search city

try:
    search_bar = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="keyword2"]')))
except TimeoutException:
    print("Timeout while locating Search Bar.\n")
else:
    search_bar.send_keys("Gurgaon")
    time.sleep(2)

# Pick first suggestion

try:
    valid_option = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="0"]')))
except TimeoutException:
    print("Timeout while locating valid search option.\n")
else:
    valid_option.click()
    time.sleep(2)

# Click Search

try:
    search_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="searchform_search_btn"]')))
except TimeoutException:
    print('Timeout while clicking on "Search" button.\n')
else:
    search_button.click()
    wait_for_page_to_load(driver, wait)

# Click "Indpedent Builder Floor" filter 

try:
    indepedent_builder_floor = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="4"]')))
except Exception:
    print('Timeout while clicking on "indepedent_builder_floor" option.\n')
else:
    indepedent_builder_floor.click()
    time.sleep(2)

# Click "Verified" filter 

try:
    verified = wait.until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                '/html[1]/body[1]/div[1]/div[1]/div[1]/div[4]/div[3]/div[1]/div[3]/section[1]/div[1]/div[1]/div[1]/div[1]/div[1]/div[1]/div[3]/span[2]',
            )
        )
    )
    verified.click()
    time.sleep(5)
except TimeoutException:
    print('Timeout while clicking on "Verified" filter.\n')


all_properties = []  
page_count = 0

while True:
    page_count += 1
    print(f"\n--- PAGE {page_count} ---")
    print("time:", time.strftime("%Y-%m-%d %H:%M:%S"))
    print("session_id:", driver.session_id)

    
    handle_popup()

    # ---- THROTTLING ----

    if page_count % 3 == 0:
        print("Throttling: sleeping 3s...")
        time.sleep(3)
    if page_count % 5 == 0:
        print("Throttling: sleeping 10s...")
        time.sleep(10)

    
    try:
        time.sleep(5)
        next_page_button = driver.find_element(By.XPATH, "//a[normalize-space()='Next Page >']")
    except NoSuchElementException:
        print(f"\n [COMPLETED] Scraping finished. Total pages navigated: {page_count}.\n")
        break
    else:
        try:
            
            driver.execute_script(
                "window.scrollBy(0, arguments[0].getBoundingClientRect().top - 100);", next_page_button
            )
            time.sleep(5)

            
            main_container = driver.find_element(By.CSS_SELECTOR, 'div[data-label="SEARCH"]')
            property_containers = main_container.find_elements(By.CLASS_NAME, "tupleNew__contentWrap")
            print(f"Number of Property Found on page {page_count}: {len(property_containers)}")

            for container in property_containers:
                property_data = {}
                try:
                    # property name
                    try:
                        property_data["property_name"] = container.find_element(By.CLASS_NAME, "tupleNew__propType").text
                    except Exception:
                        property_data["property_name"] = np.nan

                    # link 
                    try:
                        property_data["link"] = container.find_element(
                            By.CSS_SELECTOR, ".tupleNew__propertyHeading.ellipsis"
                        ).get_attribute("href")
                    except Exception:
                        property_data["link"] = np.nan

                    # society
                    try:
                        property_data["society"] = container.find_element(By.CLASS_NAME, "tupleNew__locationName").text
                    except Exception:
                        property_data["society"] = np.nan

                    # price
                    try:
                        property_data["price"] = container.find_element(By.CLASS_NAME, "tupleNew__priceValWrap").text
                    except Exception:
                        property_data["price"] = np.nan

                    # area
                    try:
                        property_data["area"] = container.find_element(
                            By.CSS_SELECTOR, "div.tupleNew__perSqftWrap.ellipsis"
                        ).text
                    except Exception:
                        property_data["area"] = np.nan

                    # area with type 
                    try:
                        area1 = container.find_element(By.CLASS_NAME, "tupleNew__area1Type").text
                    except Exception:
                        area1 = ""
                    try:
                        area2 = container.find_element(By.CLASS_NAME, "tupleNew__area2Type").text
                    except Exception:
                        area2 = ""
                    try:
                        area_type = container.find_element(By.CLASS_NAME, "tupleNew__areaType").text
                    except Exception:
                        area_type = ""

                    property_data["areawithtype"] = f"{area1} {area2} {area_type}".strip()

                    all_properties.append(property_data)

                except Exception:
                    print("Not able to extract property details!!")

            
            next_page_button = wait.until(
                EC.presence_of_element_located((By.XPATH, "//a[normalize-space()='Next Page >']"))
            )
            driver.execute_script("arguments[0].click();", next_page_button)
            print("✔ Next page clicked successfully")
            time.sleep(5)

        except Exception as e:
            print("Timeout or error while clicking on Next Page:", e)
            break

The webpage "India Real Estate Property Site - Buy Sell Rent Properties Portal - 99acres.com" did get fully loaded.

The webpage "Property in Gurgaon - Real Estate in Gurgaon" did get fully loaded.


--- PAGE 1 ---
time: 2025-10-31 00:58:15
session_id: e541972e1ba875410589a6ed593f5dbe
Number of Property Found on page 1: 78
✔ Next page clicked successfully

--- PAGE 2 ---
time: 2025-10-31 00:59:19
session_id: e541972e1ba875410589a6ed593f5dbe
Number of Property Found on page 2: 51
✔ Next page clicked successfully

--- PAGE 3 ---
time: 2025-10-31 01:00:18
session_id: e541972e1ba875410589a6ed593f5dbe
Throttling: sleeping 3s...
Number of Property Found on page 3: 26
✔ Next page clicked successfully

--- PAGE 4 ---
time: 2025-10-31 01:00:53
session_id: e541972e1ba875410589a6ed593f5dbe
Number of Property Found on page 4: 26
✔ Next page clicked successfully

--- PAGE 5 ---
time: 2025-10-31 01:01:25
session_id: e541972e1ba875410589a6ed593f5dbe
Throttling: sleeping 10s...
Number of Property Foun

In [2]:
all_properties

[{'property_name': '4 BHK Independent Builder Floor in Sushant Lok Phase 1, Gurgaon',
  'link': 'https://www.99acres.com/4-bhk-bedroom-independent-builder-floor-for-sale-in-sushant-lok-phase-1-gurgaon-2700-sq-ft-spid-O85021804',
  'society': 'Brand New Builder Floor',
  'price': '₹3.85 Cr',
  'area': '₹14,259 /sqft',
  'areawithtype': '2,700 sqft (251 sqm) Super Built-up Area'},
 {'property_name': '4 BHK Independent Builder Floor in Sector 102, Gurgaon',
  'link': 'https://www.99acres.com/4-bhk-bedroom-independent-builder-floor-for-sale-in-bptp-amstoria-sector-102-gurgaon-2000-sq-ft-spid-V85303340',
  'society': 'BPTP Amstoria',
  'price': '₹3.4 Cr',
  'area': '₹17,000 /sqft',
  'areawithtype': '2,000 sqft (186 sqm) Carpet Area'},
 {'property_name': '4 BHK Independent Builder Floor in A Block Sushant Lok Phase 1, Gurgaon',
  'link': 'https://www.99acres.com/4-bhk-bedroom-independent-builder-floor-for-sale-in-a-block-sushant-lok-phase-1-gurgaon-2650-sq-ft-spid-O83672012',
  'society': '

In [3]:
len(all_properties)

1351

In [ ]:
independent_builder_floor_df = pd.DataFrame(all_properties)
independent_builder_floor_df

In [ ]:
independent_builder_floor_df.to_csv(r"C:\Users\Jay Patel\Campusx\ml_projects\PropNavigator\data\web_scraping\independent_builder_floor_basic_details.csv", index= False)

In [ ]:
independent_builder_floor_basic_details = pd.read_csv(r"C:\Users\Jay Patel\Campusx\ml_projects\PropNavigator\data\web_scraping\independent_builder_floor_basic_details.csv")

In [7]:
independent_builder_floor_basic_details['property_id']  = independent_builder_floor_basic_details['link'].str.split("spid-").str.get(1)
independent_builder_floor_basic_details

,property_name,link,society,price,area,areawithtype,property_id
0,4 BHK Independent Builder Floor in Sushant Lok...,https://www.99acres.com/4-bhk-bedroom-independ...,Brand New Builder Floor,₹3.85 Cr,"₹14,259 /sqft","2,700 sqft (251 sqm) Super Built-up Area",O85021804
1,"4 BHK Independent Builder Floor in Sector 102,...",https://www.99acres.com/4-bhk-bedroom-independ...,BPTP Amstoria,₹3.4 Cr,"₹17,000 /sqft","2,000 sqft (186 sqm) Carpet Area",V85303340
2,4 BHK Independent Builder Floor in A Block Sus...,https://www.99acres.com/4-bhk-bedroom-independ...,"A Block Sushant Lok Phase 1, Gurgaon",₹3.66 Cr,"₹13,811 /sqft","2,650 sqft (246 sqm) Super Built-up Area",O83672012
3,4 BHK Independent Builder Floor in DLF Phase 2...,https://www.99acres.com/4-bhk-bedroom-independ...,"DLF Phase 2, Gurgaon",₹5.25 Cr,"₹19,444 /sqft","2,700 sqft (251 sqm) Carpet Area",K85623978
4,4 BHK Independent Builder Floor in DLF Phase 2...,https://www.99acres.com/4-bhk-bedroom-independ...,PARK FACING FLOOR,₹5 Cr,"₹12,500 /sqft","4,000 sqft (372 sqm) Carpet Area",E85624224
...,...,...,...,...,...,...,...
1297,"4 BHK Independent Builder Floor in Sector 46, ...",https://www.99acres.com/4-bhk-bedroom-independ...,Private Builder Floor,₹3 Cr,"₹10,000 /sqft","3,000 sqft (279 sqm) Super Built-up Area",G78860067
1298,4 BHK Independent Builder Floor in Sushant Lok...,https://www.99acres.com/4-bhk-bedroom-independ...,Sushant Lok 1 Builder Floors,₹5 Cr,"₹22,727 /sqft","2,200 sqft (204 sqm) Carpet Area",S85218742
1299,4 BHK Independent Builder Floor in Sushant Lok...,https://www.99acres.com/4-bhk-bedroom-independ...,"Sushant Lok Phase 1, Gurgaon",₹5.85 Cr,"₹20,892 /sqft","2,800 sqft (260 sqm) Carpet Area",R85216914
1300,"4 BHK Independent Builder Floor in Sector 84, ...",https://www.99acres.com/4-bhk-bedroom-independ...,SS Linden Floors,₹2.92 Cr,"₹10,428 /sqft","2,800 sqft (260 sqm) Built-up Area",B84531164


In [ ]:
# 1. HELPER FUNCTIONS

def wait_for_page_to_load(driver, timeout=15):
    WebDriverWait(driver, timeout).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )


def get_text(driver, by, value):
    try:
        return driver.find_element(by, value).text.strip()
    except:
        return np.nan


def get_list(driver, css_selector):
    try:
        items = driver.find_elements(By.CSS_SELECTOR, css_selector)
        return [i.text.strip() for i in items if i.text.strip()]
    except:
        return []


# 2. SCRAPE SINGLE PROPERTY PAGE

def scrape_details(driver, link, retries=2):
    """
    Scrapes one property detail page.
    Returns dictionary.
    """

    for _ in range(retries):
        try:
            driver.get(link)
            wait_for_page_to_load(driver)

            data = {
                "link": link,
                "bedrooms": get_text(driver, By.ID, "bedRoomNum"),
                "bathrooms": get_text(driver, By.ID, "bathroomNum"),
                "balcony": get_text(driver, By.ID, "balconyNum"),
                "additional_room": get_text(driver, By.ID, "additionalRooms"),
                "floor_info": get_text(driver, By.ID, "floorNumLabel"),
                "facing": get_text(driver, By.ID, "facingLabel"),
                "property_age": get_text(driver, By.ID, "agePossessionLbl"),
                "property_id": get_text(driver, By.ID, "Prop_Id"),
                "furnishing_details": get_list(driver, "#FurnishDetails ul#features li div"),
                "features": get_list(driver, "div[data-label='FACILITIES'] ul#features li div")
            }

            return data

        except Exception:
            time.sleep(3)

    return {"link": link}


# 3. LOAD INPUT LINKS

input_csv = r"C:\Users\Jay Patel\Campusx\ml_projects\PropNavigator\data\web_scraping\independent_builder_floor_basic_details.csv"
output_file = r"C:\Users\Jay Patel\Campusx\ml_projects\PropNavigator\data\web_scraping\independent_builder_floor_detailed_page.xlsx"

df_links = pd.read_csv(input_csv)
links = df_links["link"].dropna().tolist()
total_links = len(links)

# Load existing output 

if os.path.exists(output_file):
    existing_df = pd.read_excel(output_file)
    processed_links = set(existing_df["link"])
else:
    existing_df = pd.DataFrame()
    processed_links = set()

start = int(input("Enter link number to start from (1 for beginning): ")) - 1


# 4. SETUP DRIVER 


chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--window-size=1920,1080")
chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

driver = webdriver.Chrome(options=chrome_options)


# 5. MAIN LOOP 

BATCH_SIZE = 50

try:
    while start < total_links:
        end = min(start + BATCH_SIZE, total_links)
        print(f"\nScraping links {start + 1} to {end} of {total_links}")

        batch_data = []

        for i in range(start, end):
            link = links[i]

            if link in processed_links:
                continue

            print(f"→ Processing {i + 1}/{total_links}")
            data = scrape_details(driver, link)
            batch_data.append(data)

            time.sleep(random.uniform(2, 4))

        if batch_data:
            new_df = pd.DataFrame(batch_data)

            if not existing_df.empty:
                combined_df = (
                    pd.concat([existing_df, new_df])
                    .drop_duplicates(subset=["link"], keep="last")
                    .reset_index(drop=True)
                )
            else:
                combined_df = new_df

            combined_df.to_excel(output_file, index=False)
            existing_df = combined_df
            processed_links.update(new_df["link"].tolist())

            print(f"Saved data up to link {end}")

        start = end

        if start < total_links:
            print("Cooling down...")
            time.sleep(20)

    print("\nAll links processed successfully!")

finally:
    driver.quit()
    print("Driver closed.")



 Starting batch from link 1004 to 1053 (Total: 1351)

 Starting batch from link 1054 to 1103 (Total: 1351)
Processing link 1090 of 1351: https://www.99acres.com/4-bhk-bedroom-independent-builder-floor-for-sale-in-ansal-sushant-lok-2-sector-56-gurgaon-2000-sq-ft-spid-V84506140
Extracted details for https://www.99acres.com/4-bhk-bedroom-independent-builder-floor-for-sale-in-ansal-sushant-lok-2-sector-56-gurgaon-2000-sq-ft-spid-V84506140: {'link': 'https://www.99acres.com/4-bhk-bedroom-independent-builder-floor-for-sale-in-ansal-sushant-lok-2-sector-56-gurgaon-2000-sq-ft-spid-V84506140', 'bedrooms': '4 Bedrooms', 'bathrooms': '4 Bathrooms', 'balcony': '2 Balconies', 'additional_room': 'Pooja Room,Servant Room', 'floor_info': '2nd   of 4 Floors', 'facing': 'West', 'property_age': '0 to 1 Year Old', 'property_id': 'V84506140', 'furnishing_details': ['6 Fan', '1 Exhaust Fan', '6 Geyser', '1 Stove', '24 Light', '6 AC', '1 Modular Kitchen', '1 Chimney', '1 Curtains', '5 Wardrobe', '1 Microwav

In [5]:
indepedent_builder_floor_detailed_page = pd.read_csv(r"C:\Users\Jay Patel\OneDrive\Desktop\CampusX\ML\Projects\PropNavigator\data\web_scraping\independent_builder_floor_detailed_page.csv")
indepedent_builder_floor_detailed_page

,link,bedrooms,bathrooms,balcony,additional_room,floor_info,facing,property_age,property_id,furnishing_details,features,nearby_location
0,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Servant Room,2nd of 4 Floors,North-East,0 to 1 Year Old,O85021804,"['1 Water Purifier', '6 Fan', '1 Exhaust Fan',...","['Feng Shui / Vaastu Compliant', 'Intercom Fac...","['Bahrisons library', 'Hanuman Mandir', 'New L..."
1,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,3+ Balconies,NaN,1st of 4 Floors,North-East,1 to 5 Year Old,V85303340,"['4 Wardrobe', '5 Fan', '1 Exhaust Fan', '4 Ge...","['Feng Shui / Vaastu Compliant', 'Water purifi...","['Conscient One Mall', ""Domino's Pizza"", 'Daul..."
2,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Study Room,1st of 4 Floors,East,1 to 5 Year Old,O83672012,"['3 Fan', '1 Fridge', '5 Geyser', '1 Stove', '...","['Feng Shui / Vaastu Compliant', 'Maintenance ...","['Bahrisons library', 'Hanuman Mandir', 'New L..."
3,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,5 Bathrooms,3 Balconies,"Pooja Room,Study Room,Servant Room,Others",2nd of 4 Floors,East,1 to 5 Year Old,K85623978,"['1 Water Purifier', '12 Fan', '1 Exhaust Fan'...","['Feng Shui / Vaastu Compliant', 'Water purifi...","['Gurgaon Central Mall', 'Hanuman Mandir', 'PV..."
4,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,5 Bathrooms,2 Balconies,"Pooja Room,Study Room,Servant Room,Others",3rd of 4 Floors,East,0 to 1 Year Old,E85624224,"['11 Wardrobe', '13 Fan', '1 Exhaust Fan', '8 ...","['Feng Shui / Vaastu Compliant', 'Water purifi...","['Gurgaon Central Mall', 'PVR Cinames', 'Priva..."
...,...,...,...,...,...,...,...,...,...,...,...,...
1250,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Servant Room,1st of 4 Floors,North-East,0 to 1 Year Old,G78860067,"['8 Wardrobe', '1 Water Purifier', '12 Fan', '...","['Water purifier', 'Security / Fire Alarm', 'F...","['Dr. Aruna Kalra', 'Raheja Mall', 'Delhi Publ..."
1251,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,3 Balconies,Servant Room,Ground of 4 Floors,West,0 to 1 Year Old,S85218742,"['7 Fan', '1 Exhaust Fan', '5 Geyser', '1 Micr...","['Security / Fire Alarm', 'Intercom Facility',...","['Kotak mahindra bank ATM', 'CCD', 'Belgian Wa..."
1252,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,3+ Balconies,Servant Room,2nd of 4 Floors,East,0 to 1 Year Old,R85216914,"['4 Wardrobe', '7 Fan', '1 Exhaust Fan', '6 Ge...","['Security / Fire Alarm', 'Intercom Facility',...","['Kotak mahindra bank ATM', 'CCD', 'Belgian Wa..."
1253,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Servant Room,2nd of 4 Floors,North,0 to 1 Year Old,B84531164,"['4 Wardrobe', '4 Geyser', '6 AC', '1 Modular ...","['Water purifier', 'Centrally Air Conditioned'...","['Agri Business Management Collage', 'Manesar ..."


In [8]:
indepedent_builder_floor_merge = independent_builder_floor_basic_details.merge(indepedent_builder_floor_detailed_page, on='property_id',how= 'inner')
indepedent_builder_floor_merge

,property_name,link_x,society,price,area,areawithtype,property_id,link_y,bedrooms,bathrooms,balcony,additional_room,floor_info,facing,property_age,furnishing_details,features,nearby_location
0,4 BHK Independent Builder Floor in Sushant Lok...,https://www.99acres.com/4-bhk-bedroom-independ...,Brand New Builder Floor,₹3.85 Cr,"₹14,259 /sqft","2,700 sqft (251 sqm) Super Built-up Area",O85021804,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Servant Room,2nd of 4 Floors,North-East,0 to 1 Year Old,"['1 Water Purifier', '6 Fan', '1 Exhaust Fan',...","['Feng Shui / Vaastu Compliant', 'Intercom Fac...","['Bahrisons library', 'Hanuman Mandir', 'New L..."
1,"4 BHK Independent Builder Floor in Sector 102,...",https://www.99acres.com/4-bhk-bedroom-independ...,BPTP Amstoria,₹3.4 Cr,"₹17,000 /sqft","2,000 sqft (186 sqm) Carpet Area",V85303340,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,3+ Balconies,NaN,1st of 4 Floors,North-East,1 to 5 Year Old,"['4 Wardrobe', '5 Fan', '1 Exhaust Fan', '4 Ge...","['Feng Shui / Vaastu Compliant', 'Water purifi...","['Conscient One Mall', ""Domino's Pizza"", 'Daul..."
2,4 BHK Independent Builder Floor in A Block Sus...,https://www.99acres.com/4-bhk-bedroom-independ...,"A Block Sushant Lok Phase 1, Gurgaon",₹3.66 Cr,"₹13,811 /sqft","2,650 sqft (246 sqm) Super Built-up Area",O83672012,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Study Room,1st of 4 Floors,East,1 to 5 Year Old,"['3 Fan', '1 Fridge', '5 Geyser', '1 Stove', '...","['Feng Shui / Vaastu Compliant', 'Maintenance ...","['Bahrisons library', 'Hanuman Mandir', 'New L..."
3,4 BHK Independent Builder Floor in DLF Phase 2...,https://www.99acres.com/4-bhk-bedroom-independ...,"DLF Phase 2, Gurgaon",₹5.25 Cr,"₹19,444 /sqft","2,700 sqft (251 sqm) Carpet Area",K85623978,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,5 Bathrooms,3 Balconies,"Pooja Room,Study Room,Servant Room,Others",2nd of 4 Floors,East,1 to 5 Year Old,"['1 Water Purifier', '12 Fan', '1 Exhaust Fan'...","['Feng Shui / Vaastu Compliant', 'Water purifi...","['Gurgaon Central Mall', 'Hanuman Mandir', 'PV..."
4,4 BHK Independent Builder Floor in DLF Phase 2...,https://www.99acres.com/4-bhk-bedroom-independ...,PARK FACING FLOOR,₹5 Cr,"₹12,500 /sqft","4,000 sqft (372 sqm) Carpet Area",E85624224,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,5 Bathrooms,2 Balconies,"Pooja Room,Study Room,Servant Room,Others",3rd of 4 Floors,East,0 to 1 Year Old,"['11 Wardrobe', '13 Fan', '1 Exhaust Fan', '8 ...","['Feng Shui / Vaastu Compliant', 'Water purifi...","['Gurgaon Central Mall', 'PVR Cinames', 'Priva..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1247,"4 BHK Independent Builder Floor in Sector 46, ...",https://www.99acres.com/4-bhk-bedroom-independ...,Private Builder Floor,₹3 Cr,"₹10,000 /sqft","3,000 sqft (279 sqm) Super Built-up Area",G78860067,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Servant Room,1st of 4 Floors,North-East,0 to 1 Year Old,"['8 Wardrobe', '1 Water Purifier', '12 Fan', '...","['Water purifier', 'Security / Fire Alarm', 'F...","['Dr. Aruna Kalra', 'Raheja Mall', 'Delhi Publ..."
1248,4 BHK Independent Builder Floor in Sushant Lok...,https://www.99acres.com/4-bhk-bedroom-independ...,Sushant Lok 1 Builder Floors,₹5 Cr,"₹22,727 /sqft","2,200 sqft (204 sqm) Carpet Area",S85218742,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,3 Balconies,Servant Room,Ground of 4 Floors,West,0 to 1 Year Old,"['7 Fan', '1 Exhaust Fan', '5 Geyser', '1 Micr...","['Security / Fire Alarm', 'Intercom Facility',...","['Kotak mahindra bank ATM', 'CCD', 'Belgian Wa..."
1249,4 BHK Independent Builder Floor in Sushant Lok...,https://www.99acres.com/4-bhk-bedroom-independ...,"Sushant Lok Phase 1, Gurgaon",₹5.85 Cr,"₹20,892 /sqft","2,800 sqft (260 sqm) Carpet Area",R85216914,https://www.99acre

In [9]:
indepedent_builder_floor = indepedent_builder_floor_merge.drop(columns = ['link_x']).rename(columns ={'link_y':'link'})
indepedent_builder_floor

,property_name,society,price,area,areawithtype,property_id,link,bedrooms,bathrooms,balcony,additional_room,floor_info,facing,property_age,furnishing_details,features,nearby_location
0,4 BHK Independent Builder Floor in Sushant Lok...,Brand New Builder Floor,₹3.85 Cr,"₹14,259 /sqft","2,700 sqft (251 sqm) Super Built-up Area",O85021804,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Servant Room,2nd of 4 Floors,North-East,0 to 1 Year Old,"['1 Water Purifier', '6 Fan', '1 Exhaust Fan',...","['Feng Shui / Vaastu Compliant', 'Intercom Fac...","['Bahrisons library', 'Hanuman Mandir', 'New L..."
1,"4 BHK Independent Builder Floor in Sector 102,...",BPTP Amstoria,₹3.4 Cr,"₹17,000 /sqft","2,000 sqft (186 sqm) Carpet Area",V85303340,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,3+ Balconies,NaN,1st of 4 Floors,North-East,1 to 5 Year Old,"['4 Wardrobe', '5 Fan', '1 Exhaust Fan', '4 Ge...","['Feng Shui / Vaastu Compliant', 'Water purifi...","['Conscient One Mall', ""Domino's Pizza"", 'Daul..."
2,4 BHK Independent Builder Floor in A Block Sus...,"A Block Sushant Lok Phase 1, Gurgaon",₹3.66 Cr,"₹13,811 /sqft","2,650 sqft (246 sqm) Super Built-up Area",O83672012,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Study Room,1st of 4 Floors,East,1 to 5 Year Old,"['3 Fan', '1 Fridge', '5 Geyser', '1 Stove', '...","['Feng Shui / Vaastu Compliant', 'Maintenance ...","['Bahrisons library', 'Hanuman Mandir', 'New L..."
3,4 BHK Independent Builder Floor in DLF Phase 2...,"DLF Phase 2, Gurgaon",₹5.25 Cr,"₹19,444 /sqft","2,700 sqft (251 sqm) Carpet Area",K85623978,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,5 Bathrooms,3 Balconies,"Pooja Room,Study Room,Servant Room,Others",2nd of 4 Floors,East,1 to 5 Year Old,"['1 Water Purifier', '12 Fan', '1 Exhaust Fan'...","['Feng Shui / Vaastu Compliant', 'Water purifi...","['Gurgaon Central Mall', 'Hanuman Mandir', 'PV..."
4,4 BHK Independent Builder Floor in DLF Phase 2...,PARK FACING FLOOR,₹5 Cr,"₹12,500 /sqft","4,000 sqft (372 sqm) Carpet Area",E85624224,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,5 Bathrooms,2 Balconies,"Pooja Room,Study Room,Servant Room,Others",3rd of 4 Floors,East,0 to 1 Year Old,"['11 Wardrobe', '13 Fan', '1 Exhaust Fan', '8 ...","['Feng Shui / Vaastu Compliant', 'Water purifi...","['Gurgaon Central Mall', 'PVR Cinames', 'Priva..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1247,"4 BHK Independent Builder Floor in Sector 46, ...",Private Builder Floor,₹3 Cr,"₹10,000 /sqft","3,000 sqft (279 sqm) Super Built-up Area",G78860067,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Servant Room,1st of 4 Floors,North-East,0 to 1 Year Old,"['8 Wardrobe', '1 Water Purifier', '12 Fan', '...","['Water purifier', 'Security / Fire Alarm', 'F...","['Dr. Aruna Kalra', 'Raheja Mall', 'Delhi Publ..."
1248,4 BHK Independent Builder Floor in Sushant Lok...,Sushant Lok 1 Builder Floors,₹5 Cr,"₹22,727 /sqft","2,200 sqft (204 sqm) Carpet Area",S85218742,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,3 Balconies,Servant Room,Ground of 4 Floors,West,0 to 1 Year Old,"['7 Fan', '1 Exhaust Fan', '5 Geyser', '1 Micr...","['Security / Fire Alarm', 'Intercom Facility',...","['Kotak mahindra bank ATM', 'CCD', 'Belgian Wa..."
1249,4 BHK Independent Builder Floor in Sushant Lok...,"Sushant Lok Phase 1, Gurgaon",₹5.85 Cr,"₹20,892 /sqft","2,800 sqft (260 sqm) Carpet Area",R85216914,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,3+ Balconies,Servant Room,2nd of 4 Floors,East,0 to 1 Year Old,"['4 Wardrobe', '7 Fan', '1 Exhaust Fan', '6 Ge...","['Security / Fire Alarm', 'Intercom Facility',...","['Kotak mahindra bank ATM', 'CCD', 'Belgian Wa..."
1250,"4 BHK Independent Builder Floor in Sector 84, ...",SS Linden Floors,₹2.92 Cr,"₹10,428 /sqft","2,800 sqft (260 sqm) Built-up Area",B84

In [10]:
indepedent_builder_floor.columns = indepedent_builder_floor.columns.str.lower()

In [11]:
indepedent_builder_floor.duplicated().sum()

0

In [12]:
indepedent_builder_floor.isnull().sum()

property_name           0
society                 0
price                   0
area                    0
areawithtype            0
property_id             0
link                    0
bedrooms                0
bathrooms               0
balcony                 0
additional_room       126
floor_info              0
facing                 67
property_age            1
furnishing_details     72
features               38
nearby_location         0
dtype: int64

In [13]:
indepedent_builder_floor.to_csv(r"C:\Users\Jay Patel\OneDrive\Desktop\CampusX\ML\Projects\PropNavigator\data\web_scraping\independent_builder_floor.csv", index= False)

In [14]:
indepedent_builder_floor.head()

,property_name,society,price,area,areawithtype,property_id,link,bedrooms,bathrooms,balcony,additional_room,floor_info,facing,property_age,furnishing_details,features,nearby_location
0,4 BHK Independent Builder Floor in Sushant Lok...,Brand New Builder Floor,₹3.85 Cr,"₹14,259 /sqft","2,700 sqft (251 sqm) Super Built-up Area",O85021804,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Servant Room,2nd of 4 Floors,North-East,0 to 1 Year Old,"['1 Water Purifier', '6 Fan', '1 Exhaust Fan',...","['Feng Shui / Vaastu Compliant', 'Intercom Fac...","['Bahrisons library', 'Hanuman Mandir', 'New L..."
1,"4 BHK Independent Builder Floor in Sector 102,...",BPTP Amstoria,₹3.4 Cr,"₹17,000 /sqft","2,000 sqft (186 sqm) Carpet Area",V85303340,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,3+ Balconies,NaN,1st of 4 Floors,North-East,1 to 5 Year Old,"['4 Wardrobe', '5 Fan', '1 Exhaust Fan', '4 Ge...","['Feng Shui / Vaastu Compliant', 'Water purifi...","['Conscient One Mall', ""Domino's Pizza"", 'Daul..."
2,4 BHK Independent Builder Floor in A Block Sus...,"A Block Sushant Lok Phase 1, Gurgaon",₹3.66 Cr,"₹13,811 /sqft","2,650 sqft (246 sqm) Super Built-up Area",O83672012,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Study Room,1st of 4 Floors,East,1 to 5 Year Old,"['3 Fan', '1 Fridge', '5 Geyser', '1 Stove', '...","['Feng Shui / Vaastu Compliant', 'Maintenance ...","['Bahrisons library', 'Hanuman Mandir', 'New L..."
3,4 BHK Independent Builder Floor in DLF Phase 2...,"DLF Phase 2, Gurgaon",₹5.25 Cr,"₹19,444 /sqft","2,700 sqft (251 sqm) Carpet Area",K85623978,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,5 Bathrooms,3 Balconies,"Pooja Room,Study Room,Servant Room,Others",2nd of 4 Floors,East,1 to 5 Year Old,"['1 Water Purifier', '12 Fan', '1 Exhaust Fan'...","['Feng Shui / Vaastu Compliant', 'Water purifi...","['Gurgaon Central Mall', 'Hanuman Mandir', 'PV..."
4,4 BHK Independent Builder Floor in DLF Phase 2...,PARK FACING FLOOR,₹5 Cr,"₹12,500 /sqft","4,000 sqft (372 sqm) Carpet Area",E85624224,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,5 Bathrooms,2 Balconies,"Pooja Room,Study Room,Servant Room,Others",3rd of 4 Floors,East,0 to 1 Year Old,"['11 Wardrobe', '13 Fan', '1 Exhaust Fan', '8 ...","['Feng Shui / Vaastu Compliant', 'Water purifi...","['Gurgaon Central Mall', 'PVR Cinames', 'Priva..."
